In [ ]:
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [ ]:
X_train = pd.DataFrame({
    "fico_n": [650.0, 700.0, 750.0, 800.0],
    "dti_n": [10.0, 20.0, np.nan, 40.0],
    "revenue": [40000.0, 50000.0, 60000.0, 90000.0],
    "loan_amnt": [5000.0, 10000.0, 15000.0, 20000.0],
})

X_train

In [ ]:
imputer = SimpleImputer(strategy="median")

imputer.fit(X_train)

print("Learned medians:", imputer.statistics_)

In [ ]:
X_imputed = imputer.transform(X_train)

pd.DataFrame(
    X_imputed,
    columns=X_train.columns,
)

In [ ]:
X_validation = pd.DataFrame({
    "fico_n": [600.0, 620.0, np.nan],
    "dti_n": [50.0, np.nan, 70.0],
    "revenue": [150000.0, 200000.0, 250000.0],
    "loan_amnt": [25000.0, 30000.0, 35000.0],
})

X_validation

In [ ]:
X_validation_imputed = pd.DataFrame(
    imputer.transform(X_validation),
    columns=X_validation.columns,
)

X_validation_imputed

In [ ]:
wrong_imputer = SimpleImputer(strategy="median")

wrong_imputer.fit(X_validation)

print("Training medians:", imputer.statistics_)
print("Validation medians:", wrong_imputer.statistics_)

In [ ]:
scaler = StandardScaler()

scaler.fit(X_imputed)

print("Training means:", scaler.mean_)
print("Training standard deviations:", scaler.scale_)

In [ ]:
X_train_scaled = scaler.transform(X_imputed)

X_validation_scaled = scaler.transform(X_validation_imputed)

print("Training data:")
display(pd.DataFrame(X_train_scaled, columns=X_train.columns))

print("Validation data:")
display(pd.DataFrame(X_validation_scaled, columns=X_validation.columns))

In [ ]:
numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

X_train_processed = numerical_pipeline.fit_transform(X_train)

X_validation_processed = numerical_pipeline.transform(X_validation)

In [ ]:
np.testing.assert_allclose(
    X_train_processed,
    X_train_scaled,
)

np.testing.assert_allclose(
    X_validation_processed,
    X_validation_scaled,
)

print("Pipeline reproduces both manual transformations.")

In [ ]:
from sklearn.preprocessing import OneHotEncoder

X_train_cat = pd.DataFrame({
    "home_ownership_n": [
        "RENT", "OWN", "MORTGAGE", "RENT"
    ],
    "purpose": [
        "debt_consolidation", "credit_card",
        "debt_consolidation", "small_business"
    ],
})

X_validation_cat = pd.DataFrame({
    "home_ownership_n": [
        "RENT", "OTHER"
    ],
    "purpose": [
        "credit_card", "debt_consolidation"
    ],
})

In [ ]:
encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False,
)

X_train_encoded = encoder.fit_transform(X_train_cat)

print("Learned categories:")
for feature, categories in zip(
    X_train_cat.columns,
    encoder.categories_,
):
    print(feature, categories)

print("\nEncoded training data:")

display(
    pd.DataFrame(
        X_train_encoded,
        columns=encoder.get_feature_names_out(),
    )
)

In [ ]:
X_validation_encoded = encoder.transform(X_validation_cat)

display(
    pd.DataFrame(
        X_validation_encoded,
        columns=encoder.get_feature_names_out(),
    )
)

In [ ]:
categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="most_frequent"),
    ),
    (
        "encoder",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        ),
    ),
])

In [ ]:
X_train_cat_missing = X_train_cat.copy()

X_train_cat_missing.loc[1, "home_ownership_n"] = np.nan

X_train_cat_processed = categorical_pipeline.fit_transform(
    X_train_cat_missing
)

display(
    pd.DataFrame(
        X_train_cat_processed,
        columns=categorical_pipeline.get_feature_names_out(),
    )
)

In [ ]:
imputer = categorical_pipeline.named_steps["imputer"]
encoder = categorical_pipeline.named_steps["encoder"]

print("Imputation values:")
print(imputer.statistics_)

print("\nCategories:")
print(encoder.categories_)

print("\nOutput features:")
print(categorical_pipeline.get_feature_names_out())

In [ ]:
X_train_full = X_train.copy()

X_train_full["home_ownership_n"] = [
    "RENT", "OWN", "MORTGAGE", "RENT"
]

X_train_full["purpose"] = [
    "debt_consolidation",
    "credit_card",
    "debt_consolidation",
    "small_business",
]

display(X_train_full)

In [ ]:
from sklearn.compose import ColumnTransformer

numerical_features = [
    "fico_n",
    "dti_n",
    "revenue",
    "loan_amnt",
]

categorical_features = [
    "home_ownership_n",
    "purpose",
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numerical_pipeline,
            numerical_features,
        ),
        (
            "cat",
            categorical_pipeline,
            categorical_features,
        ),
    ],
    remainder="drop",
    verbose_feature_names_out=True,
)

In [ ]:
X_train_full_processed = preprocessor.fit_transform(
    X_train_full
)

feature_names = preprocessor.get_feature_names_out()

X_train_full_processed_df = pd.DataFrame(
    X_train_full_processed,
    columns=feature_names,
    index=X_train_full.index,
)

display(X_train_full_processed_df)

In [ ]:
print("Original shape:", X_train_full.shape)
print("Processed shape:", X_train_full_processed.shape)
print("Feature names:", feature_names)

In [ ]:
X_validation_full = X_validation.iloc[:2].copy()

X_validation_full["home_ownership_n"] = [
    "RENT", "OTHER"
]

X_validation_full["purpose"] = [
    "credit_card", "debt_consolidation"
]

X_validation_full_processed = preprocessor.transform(
    X_validation_full
)

display(
    pd.DataFrame(
        X_validation_full_processed,
        columns=preprocessor.get_feature_names_out(),
        index=X_validation_full.index,
    )
)

In [ ]:
assert X_train_full_processed.shape == (4, 10)
assert X_validation_full_processed.shape == (2, 10)

assert np.isfinite(X_train_full_processed).all()
assert np.isfinite(X_validation_full_processed).all()

print("All preprocessing checks passed.")

In [ ]:
from pathlib import Path
from underwriting.data import temporal_split

DATA_PATH = Path("../data/raw/LC_loans_granting_model_dataset.csv")

df = pd.read_csv(DATA_PATH)

splits = temporal_split(df)

for name, subset in splits.items():
    print(f"{name}: {len(subset):,} observations")

In [ ]:
numerical_features = [
    "fico_n",
    "dti_n",
    "revenue",
    "loan_amnt",
]

categorical_features = [
    "emp_length",
    "purpose",
    "home_ownership_n",
    "addr_state",
]

feature_columns = numerical_features + categorical_features

X_train = splits["train"][feature_columns].copy()
y_train = splits["train"]["Default"].copy()

X_validation = splits["validation"][feature_columns].copy()
y_validation = splits["validation"]["Default"].copy()

print("Training features:", X_train.shape)
print("Validation features:", X_validation.shape)

display(X_train.head())

In [ ]:
numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=True,
    )),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_pipeline, numerical_features),
        ("cat", categorical_pipeline, categorical_features),
    ],
    remainder="drop",
    sparse_threshold=1.0,
)

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)

print("Processed training shape:", X_train_processed.shape)
print("Output type:", type(X_train_processed))
print("Sparse output:", preprocessor.sparse_output_)

In [ ]:
feature_names = preprocessor.get_feature_names_out()

print("Number of features:", len(feature_names))

for name in feature_names:
    print(name)

In [ ]:
X_validation_processed = preprocessor.transform(X_validation)

print("Training:", X_train_processed.shape)
print("Validation:", X_validation_processed.shape)

In [ ]:
assert (
    X_train_processed.shape[1]
    == X_validation_processed.shape[1]
)

assert X_train_processed.shape[0] == len(y_train)
assert X_validation_processed.shape[0] == len(y_validation)

print("Feature dimensions are consistent.")

In [ ]:
from scipy import sparse

def check_matrix(X, name):
    values = X.data if sparse.issparse(X) else X

    assert np.isfinite(values).all(), (
        f"{name} contains non-finite values."
    )

    print(f"{name}: all values finite")


check_matrix(X_train_processed, "Training")
check_matrix(X_validation_processed, "Validation")

In [ ]:
check_matrix(X_train_processed, "Training")
check_matrix(X_validation_processed, "Validation")

In [ ]:
for name, X in [
    ("Training", X_train_processed),
    ("Validation", X_validation_processed),
]:
    print(f"\n{name}")
    print("Sparse:", sparse.issparse(X))

    if sparse.issparse(X):
        density = X.nnz / (X.shape[0] * X.shape[1])
    else:
        density = np.count_nonzero(X) / X.size

    print(f"Density: {density:.2%}")
    print(f"Sparsity: {1 - density:.2%}")

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    penalty="l2",
    C=1.0,
    solver="lbfgs",
    max_iter=1000,
    random_state=42,
)

model.fit(X_train_processed, y_train)

In [ ]:
print("Iterations:", model.n_iter_)

In [ ]:
p_train = model.predict_proba(X_train_processed)[:, 1]

p_validation = model.predict_proba(
    X_validation_processed
)[:, 1]

print("First five validation probabilities:")
print(p_validation[:5])

In [ ]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss,
)

def evaluate_predictions(y, p):
    return {
        "ROC-AUC": roc_auc_score(y, p),
        "Average precision": average_precision_score(y, p),
        "Log loss": log_loss(y, p),
    }


results = pd.DataFrame({
    "Training": evaluate_predictions(y_train, p_train),
    "Validation": evaluate_predictions(
        y_validation, p_validation
    ),
})

display(results)

In [ ]:
baseline_probability = y_train.mean()

p_baseline = np.full(
    len(y_validation),
    baseline_probability,
)

print("Training default rate:", baseline_probability)
print("Validation default rate:", y_validation.mean())

print(
    "Baseline log loss:",
    log_loss(y_validation, p_baseline),
)

print(
    "Model log loss:",
    log_loss(y_validation, p_validation),
)

In [ ]:
import matplotlib.pyplot as plt

from sklearn.metrics import RocCurveDisplay

RocCurveDisplay.from_predictions(
    y_validation,
    p_validation,
)

plt.title("Logistic regression — 2016 validation")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
from sklearn.metrics import log_loss

train_default_rate = y_train.mean()
validation_default_rate = y_validation.mean()

p_constant = np.full(
    len(y_validation),
    train_default_rate,
)

baseline_log_loss = log_loss(
    y_validation,
    p_constant,
)

model_log_loss = log_loss(
    y_validation,
    p_validation,
)

print(f"Training default rate:   {train_default_rate:.4f}")
print(f"Validation default rate: {validation_default_rate:.4f}")
print(f"Constant baseline loss:  {baseline_log_loss:.4f}")
print(f"Logistic regression loss:{model_log_loss:.4f}")

In [ ]:
oracle_constant_loss = log_loss(
    y_validation,
    np.full(len(y_validation), validation_default_rate),
)

print(f"Oracle constant loss: {oracle_constant_loss:.4f}")